In [0]:
from pyspark.sql import functions as F

STORAGE = "stolistdl2026"
BRONZE = f"abfss://bronze@{STORAGE}.dfs.core.windows.net/olist/"
CATALOG, SCHEMA = "olist_dev", "silver"

def read_bronze(filename):
    """Bronze CSV padho. Sab columns string rahenge, types hum khud lagayenge."""
    return (spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(BRONZE + filename)
        .withColumn("source_file", F.col("_metadata.file_path")))

def clean_strings(df):
    """Har string column ko trim karo, aur khali string ko NULL banao."""
    for c, t in df.dtypes:
        if t == "string" and c != "source_file":
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))))
    return df

def apply_types(df, casts):
    for col, dtype in casts.items():
        if dtype == "timestamp":
            df = df.withColumn(col, F.to_timestamp(F.col(col)))
        else:
            df = df.withColumn(col, F.col(col).cast(dtype))
    return df

def write_silver(df, table):
    df = df.withColumn("ingestion_ts", F.current_timestamp())
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    print(f"{table}: {spark.table(f'{CATALOG}.{SCHEMA}.{table}').count()} rows")

In [0]:
TABLES = {
    "customers": {
        "file": "olist_customers_dataset.csv",
        "keys": ["customer_id"],
        "casts": {}
    },
    "orders": {
        "file": "olist_orders_dataset.csv",
        "keys": ["order_id"],
        "casts": {c: "timestamp" for c in [
            "order_purchase_timestamp", "order_approved_at",
            "order_delivered_carrier_date", "order_delivered_customer_date",
            "order_estimated_delivery_date"]}
    },
    "order_items": {
        "file": "olist_order_items_dataset.csv",
        "keys": ["order_id", "order_item_id"],
        "casts": {"order_item_id": "int", "shipping_limit_date": "timestamp",
                  "price": "double", "freight_value": "double"}
    },
    "order_payments": {
        "file": "olist_order_payments_dataset.csv",
        "keys": ["order_id", "payment_sequential"],
        "casts": {"payment_sequential": "int", "payment_installments": "int",
                  "payment_value": "double"}
    },
    "order_reviews": {
        "file": "olist_order_reviews_dataset.csv",
        "keys": ["review_id", "order_id"],
        "casts": {"review_score": "int", "review_creation_date": "timestamp",
                  "review_answer_timestamp": "timestamp"}
    },
    "products": {
        "file": "olist_products_dataset.csv",
        "keys": ["product_id"],
        "casts": {"product_name_length": "int", "product_description_length": "int",
                  "product_photos_qty": "int", "product_weight_g": "double",
                  "product_length_cm": "double", "product_height_cm": "double",
                  "product_width_cm": "double"}
    },
    "sellers": {
        "file": "olist_sellers_dataset.csv",
        "keys": ["seller_id"],
        "casts": {}
    },
    "category_translation": {
        "file": "product_category_name_translation.csv",
        "keys": ["product_category_name"],
        "casts": {}
    },
}

In [0]:
import uuid
dbutils.widgets.text("run_id", "")
RUN_ID = dbutils.widgets.get("run_id") or str(uuid.uuid4())

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.quarantine (
  run_id STRING, table_name STRING, failed_rules STRING,
  row_json STRING, quarantined_ts TIMESTAMP)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.dq_results (
  run_id STRING, table_name STRING, rule_name STRING, severity STRING,
  total_rows BIGINT, failed_rows BIGINT, failed_pct DOUBLE, run_ts TIMESTAMP)
""")
print("Run ID:", RUN_ID)

Run ID: 5b0d4c73-76fe-4c1d-a2c7-4210e9d61114


In [0]:
RULES = {
    "customers": [
        ("customer_id_not_null", "customer_id IS NOT NULL", "error"),
        ("zip_not_null", "customer_zip_code_prefix IS NOT NULL", "error"),
        ("state_has_2_chars", "length(customer_state) = 2", "warn"),
    ],
    "orders": [
        ("order_id_not_null", "order_id IS NOT NULL", "error"),
        ("customer_id_not_null", "customer_id IS NOT NULL", "error"),
        ("purchase_ts_not_null", "order_purchase_timestamp IS NOT NULL", "error"),
        ("valid_status", "order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled')", "error"),
        # Undelivered orders mein delivery date NULL hoti hai, wo valid hai
        ("delivery_after_purchase", "order_delivered_customer_date IS NULL OR order_delivered_customer_date >= order_purchase_timestamp", "error"),
    ],
    "order_items": [
        ("order_id_not_null", "order_id IS NOT NULL", "error"),
        ("product_id_not_null", "product_id IS NOT NULL", "error"),
        ("price_non_negative", "price >= 0", "error"),
        ("freight_non_negative", "freight_value >= 0", "error"),
    ],
    "order_payments": [
        ("order_id_not_null", "order_id IS NOT NULL", "error"),
        ("payment_value_non_negative", "payment_value >= 0", "error"),
        ("payment_type_known", "payment_type <> 'not_defined'", "warn"),
    ],
    "order_reviews": [
        ("review_id_not_null", "review_id IS NOT NULL", "error"),
        ("order_id_not_null", "order_id IS NOT NULL", "error"),
        ("score_between_1_5", "review_score BETWEEN 1 AND 5", "error"),
    ],
    "products": [
        ("product_id_not_null", "product_id IS NOT NULL", "error"),
        ("weight_positive", "product_weight_g IS NULL OR product_weight_g > 0", "warn"),
    ],
    "sellers": [
        ("seller_id_not_null", "seller_id IS NOT NULL", "error"),
    ],
    "category_translation": [
        ("category_not_null", "product_category_name IS NOT NULL", "error"),
    ],
}

In [0]:
def apply_dq(df, table):
    """(good_df, quarantine_df) return karta hai aur rule results log karta hai."""
    rules = RULES.get(table, [])
    data_cols = df.columns

    # Har rule ke liye ek flag column: True = row is rule mein FAIL hui
    # coalesce(..., False) isliye ki NULL result bhi fail maana jaye
    flagged = df
    for name, expr, sev in rules:
        flagged = flagged.withColumn(f"_f_{name}", ~F.coalesce(F.expr(expr), F.lit(False)))
    flagged = flagged.cache()
    total = flagged.count()

    # Ek hi pass mein har rule ke fail counts
    counts = (flagged.agg(*[F.sum(F.col(f"_f_{n}").cast("int")).alias(n) for n, _, _ in rules])
                     .collect()[0].asDict())

    # Results log karo
    log_rows = []
    for n, _, sev in rules:
        failed = int(counts[n] or 0)
        log_rows.append((RUN_ID, table, n, sev, total, failed, round(100.0 * failed / max(total, 1), 4)))
    (spark.createDataFrame(log_rows,
        "run_id string, table_name string, rule_name string, severity string, "
        "total_rows long, failed_rows long, failed_pct double")
     .withColumn("run_ts", F.current_timestamp())
     .write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.dq_results"))

    # Sirf 'error' rules se good/bad split
    err_rules = [n for n, _, sev in rules if sev == "error"]
    failed_arr = F.filter(
        F.array(*[F.when(F.col(f"_f_{n}"), F.lit(n)) for n in err_rules]),
        lambda x: x.isNotNull())
    flagged = flagged.withColumn("_failed", failed_arr)

    good = flagged.filter(F.size("_failed") == 0).select(*data_cols)
    bad = (flagged.filter(F.size("_failed") > 0)
           .select(F.lit(RUN_ID).alias("run_id"),
                   F.lit(table).alias("table_name"),
                   F.concat_ws(",", "_failed").alias("failed_rules"),
                   F.to_json(F.struct(*data_cols)).alias("row_json"),
                   F.current_timestamp().alias("quarantined_ts")))
    return good, bad, flagged

In [0]:
def apply_dq(df, table):
    """(good_df, quarantine_df, flagged_df) return karta hai aur rule results log karta hai."""
    rules = RULES.get(table, [])
    data_cols = df.columns

    # Har rule ke liye ek flag column: True = row is rule mein FAIL hui
    flagged = df
    for name, expr, sev in rules:
        flagged = flagged.withColumn(f"_f_{name}", ~F.coalesce(F.expr(expr), F.lit(False)))
    total = flagged.count()

    # Ek hi pass mein har rule ke fail counts
    counts = (flagged.agg(*[F.sum(F.col(f"_f_{n}").cast("int")).alias(n) for n, _, _ in rules])
                     .collect()[0].asDict())

    # Results log karo
    log_rows = []
    for n, _, sev in rules:
        failed = int(counts[n] or 0)
        log_rows.append((RUN_ID, table, n, sev, total, failed, round(100.0 * failed / max(total, 1), 4)))
    (spark.createDataFrame(log_rows,
        "run_id string, table_name string, rule_name string, severity string, "
        "total_rows long, failed_rows long, failed_pct double")
     .withColumn("run_ts", F.current_timestamp())
     .write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.dq_results"))

    # Sirf 'error' rules se good/bad split
    err_rules = [n for n, _, sev in rules if sev == "error"]
    failed_arr = F.filter(
        F.array(*[F.when(F.col(f"_f_{n}"), F.lit(n)) for n in err_rules]),
        lambda x: x.isNotNull())
    flagged = flagged.withColumn("_failed", failed_arr)

    good = flagged.filter(F.size("_failed") == 0).select(*data_cols)
    bad = (flagged.filter(F.size("_failed") > 0)
           .select(F.lit(RUN_ID).alias("run_id"),
                   F.lit(table).alias("table_name"),
                   F.concat_ws(",", "_failed").alias("failed_rules"),
                   F.to_json(F.struct(*data_cols)).alias("row_json"),
                   F.current_timestamp().alias("quarantined_ts")))
    return good, bad, flagged

In [0]:
for table, cfg in TABLES.items():
    df = read_bronze(cfg["file"])

    if table == "products":
        df = (df.withColumnRenamed("product_name_lenght", "product_name_length")
                .withColumnRenamed("product_description_lenght", "product_description_length"))

    df = clean_strings(df)
    df = apply_types(df, cfg["casts"])
    df = df.dropDuplicates(cfg["keys"])

    good, bad, flagged = apply_dq(df, table)

    # Idempotent: is table ki purani quarantine rows hata ke naye likho
    spark.sql(f"DELETE FROM {CATALOG}.{SCHEMA}.quarantine WHERE table_name = '{table}'")
    bad.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.quarantine")

    write_silver(good, table)
    print(f"   quarantined: {bad.count()}")

customers: 99441 rows
   quarantined: 0
orders: 99441 rows
   quarantined: 0
order_items: 112650 rows
   quarantined: 0
order_payments: 103886 rows
   quarantined: 0
order_reviews: 99224 rows
   quarantined: 0
products: 32951 rows
   quarantined: 0
sellers: 3095 rows
   quarantined: 0
category_translation: 71 rows
   quarantined: 0


In [0]:
def ri_check(child, ckey, parent, pkey):
    c = spark.table(f"{CATALOG}.{SCHEMA}.{child}")
    p = spark.table(f"{CATALOG}.{SCHEMA}.{parent}").select(pkey).distinct()
    total = c.count()
    orphans = c.join(p, c[ckey] == p[pkey], "left_anti").count()
    rule = f"ri_{child}.{ckey}->{parent}.{pkey}"
    (spark.createDataFrame(
        [(RUN_ID, child, rule, "warn", total, orphans, round(100.0 * orphans / max(total, 1), 4))],
        "run_id string, table_name string, rule_name string, severity string, "
        "total_rows long, failed_rows long, failed_pct double")
     .withColumn("run_ts", F.current_timestamp())
     .write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.dq_results"))
    print(f"{rule}: {orphans} orphan rows out of {total}")

ri_check("orders", "customer_id", "customers", "customer_id")
ri_check("order_items", "order_id", "orders", "order_id")
ri_check("order_items", "product_id", "products", "product_id")
ri_check("order_items", "seller_id", "sellers", "seller_id")
ri_check("order_payments", "order_id", "orders", "order_id")
ri_check("order_reviews", "order_id", "orders", "order_id")

ri_orders.customer_id->customers.customer_id: 0 orphan rows out of 99441
ri_order_items.order_id->orders.order_id: 0 orphan rows out of 112650
ri_order_items.product_id->products.product_id: 0 orphan rows out of 112650
ri_order_items.seller_id->sellers.seller_id: 0 orphan rows out of 112650
ri_order_payments.order_id->orders.order_id: 0 orphan rows out of 103886
ri_order_reviews.order_id->orders.order_id: 0 orphan rows out of 99224


In [0]:
geo = read_bronze("olist_geolocation_dataset.csv")
geo = clean_strings(geo)
geo = (geo
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
    .groupBy("geolocation_zip_code_prefix")
    .agg(F.avg("geolocation_lat").alias("lat"),
         F.avg("geolocation_lng").alias("lng"),
         F.first("geolocation_city", ignorenulls=True).alias("city"),
         F.first("geolocation_state", ignorenulls=True).alias("state"),
         F.first("source_file").alias("source_file")))
write_silver(geo, "geolocation")

geolocation: 19015 rows


In [0]:
%sql
SELECT table_name, rule_name, severity, total_rows, failed_rows, failed_pct
FROM olist_dev.silver.dq_results
WHERE run_id = (SELECT run_id FROM olist_dev.silver.dq_results ORDER BY run_ts DESC LIMIT 1)
ORDER BY failed_rows DESC;

table_name,rule_name,severity,total_rows,failed_rows,failed_pct
products,weight_positive,warn,32951,4,0.0121
order_payments,payment_type_known,warn,103886,3,0.0029
category_translation,category_not_null,error,71,0,0.0
order_reviews,ri_order_reviews.order_id->orders.order_id,warn,99224,0,0.0
orders,ri_orders.customer_id->customers.customer_id,warn,99441,0,0.0
products,product_id_not_null,error,32951,0,0.0
customers,state_has_2_chars,warn,99441,0,0.0
order_items,price_non_negative,error,112650,0,0.0
customers,customer_id_not_null,error,99441,0,0.0
order_items,ri_order_items.product_id->products.product_id,warn,112650,0,0.0


In [0]:
%sql
SELECT table_name, failed_rules, COUNT(*) AS cnt
FROM olist_dev.silver.quarantine
GROUP BY table_name, failed_rules;

table_name,failed_rules,cnt


In [0]:
from pyspark.sql import Row

test_df = read_bronze("olist_order_reviews_dataset.csv")
test_df = apply_types(clean_strings(test_df), TABLES["order_reviews"]["casts"])

fake = spark.createDataFrame([
    Row(review_id="fake1", order_id="fake_order", review_score=9),
    Row(review_id=None,    order_id="fake_order", review_score=3),
]).withColumn("review_comment_title", F.lit(None).cast("string")) \
  .withColumn("review_comment_message", F.lit(None).cast("string")) \
  .withColumn("review_creation_date", F.lit(None).cast("timestamp")) \
  .withColumn("review_answer_timestamp", F.lit(None).cast("timestamp")) \
  .withColumn("source_file", F.lit("test"))

test_df = test_df.unionByName(fake, allowMissingColumns=True)
good, bad, _ = apply_dq(test_df, "order_reviews")
display(bad.select("failed_rules", "row_json"))

failed_rules,row_json
score_between_1_5,"{""review_id"":""fake1"",""order_id"":""fake_order"",""review_score"":9,""source_file"":""test""}"
review_id_not_null,"{""order_id"":""fake_order"",""review_score"":3,""source_file"":""test""}"


In [0]:
import json
quar = spark.sql(f"SELECT COUNT(*) c FROM {CATALOG}.{SCHEMA}.quarantine WHERE run_id = '{RUN_ID}'").first()["c"]
print("Quarantined rows this run:", quar)

# Production guard: zyada kharab data ho toh pipeline fail ho
if quar > 5000:
    raise Exception(f"Too many quarantined rows: {quar}")

dbutils.notebook.exit(json.dumps({"run_id": RUN_ID, "quarantined": quar}))